# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL from [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

> This notebook demonstrates how to access, explore and process this data using [mlcroissant](https://github.com/mlcommons/croissant) and pandas.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Convert metadata to a dict for pretty-printing
metadata = dataset.metadata.to_json()
print(metadata['name'] + ':')
print(metadata['description'])

## 2. Data Overview
Review available record sets, their `@id`s, and all fields (columns) they include.

The `mlcroissant` API lets us list record sets and the fields (columns) for each. For complex Croissant datasets, there may be several record sets representing different data tables, each with an `@id` and set of fields referenced by their `@id`s.

Let's enumerate all record sets and their fields with `@id`s to understand what we can query.

In [ ]:
# List all record sets in the dataset, along with their @ids and field IDs
print('Record sets and their fields:')
record_sets = dataset.metadata.record_sets
all_record_sets = []

for rs in record_sets:
    print(f"- Record set: {rs.name} (@id: {rs.id})")
    all_record_sets.append(rs.id)
    print('  Fields:')
    for field in rs.fields:
        col_list = ', '.join([col.id for col in field.columns]) if field.columns else "-"
        print(f"    - {field.name} (@id: {field.id}, columns: {col_list})")
    print('')
if not all_record_sets:
    print('No explicit record sets listed in the top-level Croissant metadata, attempting automatic inference by inspecting the data files ...')
    # mlcroissant will automatically infer a record set if only a single tabular file is distributed
    # Let's try to load file listings via distribution
    for file in dataset.metadata.distributions:
        print(f"  File: {file.name} (@id: {file.id}) columns (fields):")
        for field in file.fields:
            print(f"    - {field.name} (@id: {field.id})")
        print('')

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis. All record set identifiers (`@id`) are used directly (not names).

By default, `mlcroissant` supports iterating records for a record set by `@id`, or if there is only one record set (or a single main file), that will often be the default. Here we demonstrate how to extract all available record sets.


In [ ]:
# Find record sets (by @id). If none, try to extract from main distribution as an implicit record set.
if all_record_sets:
    record_sets_ids = all_record_sets
else:
    # Fallback: assume main file distribution is a record set
    record_sets_ids = [dataset.metadata.distributions[0].id]
    print('Using the dataset distribution as default record set.')

dataframes = {}
for record_set_id in record_sets_ids:
    # Records may be loaded via record_set argument by @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id} with shape {df.shape}")

# Show columns from the first record set
main_record_set = record_sets_ids[0]
print(f'Columns for record set {main_record_set}:')
print(dataframes[main_record_set].columns.tolist())
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Now, let's do some basic EDA with the main record set. We will choose an appropriate numeric field (such as protein abundance, peptide count, or MW) by its `@id`.

- We'll filter records where a numeric column exceeds a threshold,
- Normalize this field (z-score),
- Optionally, group by a categorical field and compute mean values.

In [ ]:
# Choose a numeric field by @id. Let us try some common column names or inspect columns if needed.
numeric_field_candidates = ['abundance', 'MW', 'peptide_count', 'coverage']

df = dataframes[main_record_set].copy()
available_columns = df.columns.tolist()
chosen_field = None
for candidate in numeric_field_candidates:
    # Check for common data naming issues
    if candidate in available_columns:
        chosen_field = candidate
        break
    # Snake/camelCase variations
    if candidate.lower() in [c.lower() for c in available_columns]:
        chosen_field = [c for c in available_columns if c.lower()==candidate.lower()][0]
        break
if not chosen_field:
    # Fallback: pick first float/integer-like column
    for col in available_columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            chosen_field = col
            break
print(f"Using numeric field '{chosen_field}' for EDA.")

threshold = df[chosen_field].quantile(0.75) if chosen_field else 0
filtered_df = df[df[chosen_field] > threshold]
print(f"Filtered records with '{chosen_field}' > {threshold:.3f}:")
display(filtered_df.head())

filtered_df[f"{chosen_field}_normalized"] = (
    filtered_df[chosen_field] - filtered_df[chosen_field].mean()
) / filtered_df[chosen_field].std()

print(f"Normalized '{chosen_field}' for filtered records:")
display(filtered_df[[chosen_field, f"{chosen_field}_normalized"]].head())

# Group by a categorical field, e.g., 'description' or 'sample_id'
group_field_candidates = ['description', 'sample_id', 'accession']
group_field = None
for candidate in group_field_candidates:
    if candidate in filtered_df.columns:
        group_field = candidate
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by '{group_field}':")
    display(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and, if possible, compare group means if a categorical field is available.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[chosen_field], kde=True, bins=30)
plt.title(f"Distribution of '{chosen_field}' across all records")
plt.xlabel(chosen_field)
plt.ylabel('Count')
plt.show()

if group_field:
    plt.figure(figsize=(12, 4))
    # Plot mean normalized by group
    group_means = grouped_df[f"{chosen_field}_normalized"] if f"{chosen_field}_normalized" in grouped_df else grouped_df[chosen_field]
    group_means = group_means.sort_values(ascending=False)
    group_means.plot(kind='bar')
    plt.title(f"Mean normalized '{chosen_field}' by '{group_field}' group")
    plt.ylabel('Mean normalized value')
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- We loaded a detailed Croissant metadata-backed mass spectrometry dataset using `mlcroissant`.
- Explored record sets and their fields, extracting data by their unique `@id`.
- Performed simple EDA by filtering and normalizing protein-level abundance or other numeric fields.
- Visualized distribution and (where available) summarized by categorical groups.

Use this workflow as a template for FAIR data access and inspection with Croissant schema datasets!
